In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv("toy_vampseq_bin_counts.csv")

# explain what each column means
df.head(5)

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count
0,WT,wild_type,1,9,42,267,376
1,p.Gly12Gly,synonymous,1,12,32,247,316
2,p.Trp18Ter,nonsense,1,221,110,41,6
3,p.Gln36Ter,nonsense,1,217,115,38,11
4,p.Tyr52Ter,nonsense,1,207,89,30,7


| column | meaning |
|---|---|
| `variant` | Variant name (protein change - ex. `p.Leu25Pro` = Leu at position 25 -> Pro). |
| `variant_type` | Type of variant category: `wild_type`, `synonymous`, `nonsense`, `missense`. |
| `replicate` | Independent repeat of the sort-and-sequence experiment: 1 or 2. |
| `bin1_count` .. `bin4_count` | Reads for that variant in each sorted fluorescence bin. bin1 = dimmest/lowest abundance, bin4 = brightest/highest abundance. |

In [30]:
print(df.shape)
print(df.dtypes)

(24, 7)
variant           str
variant_type      str
replicate       int64
bin1_count      int64
bin2_count      int64
bin3_count      int64
bin4_count      int64
dtype: object


## Step 1

In [31]:
# bins and weight constants 
bin_cols = ["bin1_count", "bin2_count", "bin3_count", "bin4_count"]
# normalized weights
weights = np.array([0.25, 0.5, 0.75, 1.0])

# Step 1: frequency = variant's count in a bin / total reads in that bin
bin_cols = ["bin1_count", "bin2_count", "bin3_count", "bin4_count"]

# total reads in that bin
## bin_totals = df[bin_cols].sum()
bin_totals = df.groupby("replicate")[bin_cols].transform("sum")
# variant's count in a bin / total reads in that bin
freq = df[bin_cols].div(bin_totals)

# add the frequencies into df
freq_cols = [c.replace("_count", "_freq") for c in bin_cols]
df[freq_cols] = freq

df

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375


## Step 2

In [32]:
# Step 2: weighted score per variant
weights = np.array([0.25, 0.5, 0.75, 1.0])

# add the weighted scores into df
df["W"] = freq.dot(weights) / freq.sum(axis=1)

df

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq,W
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000,0.843642
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500,0.837455
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750,0.387393
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875,0.393937
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375,0.376158
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875,0.434735
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500,0.526221
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875,0.622872
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125,0.598319
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375,0.810591


## Step 3

In [33]:
# Step 3: normalize using nonsense (floor) and wild-type (ceiling)
# computed per replicate, since bin totals differ between experiments

## nonsense_W_median = df.loc[df.variant_type == "nonsense", "W"].median()
nonsense_W_median = df[df.variant_type == "nonsense"].groupby("replicate")["W"].median()
## wt_W = df.loc[df.variant_type == "wild_type", "W"].median()  # adjust label to match your data
wt_W = df[df.variant_type == "wild_type"].groupby("replicate")["W"].first()

## df["abundance_score"] = (df["W"] - nonsense_W_median) / (wt_W - nonsense_W_median)
df["abundance_score"] = (df["W"] - df["replicate"].map(nonsense_W_median)) / (df["replicate"].map(wt_W) - df["replicate"].map(nonsense_W_median))

# check: WT should land at exactly 1 in every replicate now
assert np.allclose(df.loc[df.variant_type == "wild_type", "abundance_score"], 1.0)

# save as a new CSV
df.to_csv("final.csv", index=False)

df

,variant,variant_type,replicate,bin1_count,bin2_count,bin3_count,bin4_count,bin1_freq,bin2_freq,bin3_freq,bin4_freq,W,abundance_score
0,WT,wild_type,1,9,42,267,376,0.007500,0.042000,0.190714,0.235000,0.843642,1.000000
1,p.Gly12Gly,synonymous,1,12,32,247,316,0.010000,0.032000,0.176429,0.197500,0.837455,0.986439
2,p.Trp18Ter,nonsense,1,221,110,41,6,0.184167,0.110000,0.029286,0.003750,0.387393,0.000000
3,p.Gln36Ter,nonsense,1,217,115,38,11,0.180833,0.115000,0.027143,0.006875,0.393937,0.014342
4,p.Tyr52Ter,nonsense,1,207,89,30,7,0.172500,0.089000,0.021429,0.004375,0.376158,-0.024626
5,p.Leu25Pro,missense,1,190,154,68,11,0.158333,0.154000,0.048571,0.006875,0.434735,0.103764
6,p.Val30Asp,missense,1,124,208,146,36,0.103333,0.208000,0.104286,0.022500,0.526221,0.304282
7,p.Ile40Asn,missense,1,40,172,258,43,0.033333,0.172000,0.184286,0.026875,0.622872,0.516120
8,p.Gly46Arg,missense,1,159,17,22,181,0.132500,0.017000,0.015714,0.113125,0.598319,0.462305
9,p.Ser45Thr,missense,1,17,54,214,287,0.014167,0.054000,0.152857,0.179375,0.810591,0.927559
